In [ ]:
# Library import

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import requests
from bs4 import BeautifulSoup

# Plot styling
sns.set_style('ticks')

### 1. Dataset Loading

In [16]:
# Read tracks.csv
tracks = pd.read_csv('../data/kaggle2/tracks.csv')
tracks.head()

,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.4450,0,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.2630,0,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.1770,1,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.321,0.0946,7,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922,0.402,0.1580,3,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4


### 2. Data Overview

In [17]:
tracks.info()

<class 'pandas.DataFrame'>
RangeIndex: 586672 entries, 0 to 586671
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                586672 non-null  str    
 1   name              586601 non-null  str    
 2   popularity        586672 non-null  int64  
 3   duration_ms       586672 non-null  int64  
 4   explicit          586672 non-null  int64  
 5   artists           586672 non-null  str    
 6   id_artists        586672 non-null  str    
 7   release_date      586672 non-null  str    
 8   danceability      586672 non-null  float64
 9   energy            586672 non-null  float64
 10  key               586672 non-null  int64  
 11  loudness          586672 non-null  float64
 12  mode              586672 non-null  int64  
 13  speechiness       586672 non-null  float64
 14  acousticness      586672 non-null  float64
 15  instrumentalness  586672 non-null  float64
 16  liveness          586672 non-nu

In [18]:
# Checking for null values 
tracks.isnull().sum()

id                   0
name                71
popularity           0
duration_ms          0
explicit             0
artists              0
id_artists           0
release_date         0
danceability         0
energy               0
key                  0
loudness             0
mode                 0
speechiness          0
acousticness         0
instrumentalness     0
liveness             0
valence              0
tempo                0
time_signature       0
dtype: int64

### 3. Data Cleaning

In [19]:
# Handle missing names
tracks['name'] = tracks['name'].fillna("Unknown Track")

**DEALING WITH "STRINGIFIED" LISTS:**

The `artists` and `id_artists` columns were imported as stringified lists. Because Python interprets these as literal strings rather than list objects, individual elements are not directly accessible for analysis.

In [20]:
print(f"""      --- 'artists' column ---
{tracks['artists'].tail()}

      --- 'id_artists' column ---
{tracks['id_artists'].tail()}""")

      --- 'artists' column ---
586667                        ['阿YueYue']
586668                     ['ROLE MODEL']
586669                        ['FINNEAS']
586670    ['Gentle Bones', 'Clara Benin']
586671                      ['Afrosound']
Name: artists, dtype: str

      --- 'id_artists' column ---
586667                           ['1QLBXKM5GCpyQQSVMNZqrZ']
586668                           ['1dy5WNgIKQU6ezkpZs4y8z']
586669                           ['37M5pPGs6V1fchFJSgCguX']
586670    ['4jGPdu95icCKVF31CcFKbS', '5ebPSE9YI5aLeZ1Z2g...
586671                           ['0i4Qda0k4nf7jnNHmSNpYv']
Name: id_artists, dtype: str


In [21]:
# 1. Clean the string by removing list symbols
tracks['artists'] = tracks['artists'].str.replace("[", "", regex=False).str.replace("]", "", regex=False).str.replace("'", "", regex=False)
tracks['id_artists'] = tracks['id_artists'].str.replace("[", "", regex=False).str.replace("]", "", regex=False).str.replace("'", "", regex=False)

# 2. Split the string to create a list
tracks['artists'] = tracks['artists'].str.split(", ")
tracks['id_artists'] = tracks['id_artists'].str.split(", ")

# 3. Extract the first element (Main Artist)
tracks['main_artist'] = tracks['artists'].str[0]
tracks['id_main_artist'] = tracks['id_artists'].str[0]

In [22]:
print(f"""      --- 'artists' column ---
{tracks['artists'].tail()}

      --- 'id_artists' column ---
{tracks['id_artists'].tail()}""")

      --- 'artists' column ---
586667                      [阿YueYue]
586668                   [ROLE MODEL]
586669                      [FINNEAS]
586670    [Gentle Bones, Clara Benin]
586671                    [Afrosound]
Name: artists, dtype: object

      --- 'id_artists' column ---
586667                            [1QLBXKM5GCpyQQSVMNZqrZ]
586668                            [1dy5WNgIKQU6ezkpZs4y8z]
586669                            [37M5pPGs6V1fchFJSgCguX]
586670    [4jGPdu95icCKVF31CcFKbS, 5ebPSE9YI5aLeZ1Z2gkqjn]
586671                            [0i4Qda0k4nf7jnNHmSNpYv]
Name: id_artists, dtype: object


**Handling Inconsistent Formats:**

- This Spotify dataset presents inconsistent date formats. In some case full dates (**YYYY-MM-DD**) are present, in others year-only entries (**YYYY**). Not knowing which other formats are present, we used this code to filter out any value shorter than the standard 10 characters.
- By extracting the unique values from this subset, we were able to identify exactly which patterns were present (such as **YYYY** or **YYYY-MM**).

In [23]:
tracks.release_date

0         1922-02-22
1         1922-06-01
2         1922-03-21
3         1922-03-21
4               1922
             ...    
586667    2020-09-26
586668    2020-10-21
586669    2020-09-02
586670    2021-03-05
586671    2015-07-01
Name: release_date, Length: 586672, dtype: str

In [24]:
# Create a mask for strings with lengths shorter than a full date (10 characters)
# This helps us identify years (length 4) and month-year formats (length 7)
anomaly_mask = (tracks['release_date'].str.len() < 10)

# Display unique values to understand the specific formats present
anomalous_values = tracks.loc[anomaly_mask, 'release_date'].unique()
print(anomalous_values)

<StringArray>
[   '1922',    '1923',    '1924',    '1925',    '1926',    '1927',    '1928',
    '1929',    '1930',    '1931',
 ...
 '1973-01', '1976-03', '1981-05', '1983-09', '1985-04', '1996-03', '1980-10',
 '1981-10', '1999-10', '1991-05']
Length: 340, dtype: str


In [25]:
# --- STEP 1: Handle year-only dates (YYYY) ---
# Check for strings with 4 characters and set them to 01-01
mask_year = tracks['release_date'].str.len() == 4
tracks.loc[mask_year, 'release_date'] = tracks.loc[mask_year, 'release_date'] + '-01-01'

# --- STEP 2: Handle year-month dates (YYYY-MM) ---
# Check for strings with 7 characters and set them to the 01 day of that month
mask_month = tracks['release_date'].str.len() == 7
tracks.loc[mask_month, 'release_date'] = tracks.loc[mask_month, 'release_date'] + '-01'

# --- STEP 3: Final Conversion ---
# Now that formats are standardized to YYYY-MM-DD, convert to datetime objects
tracks['release_date'] = pd.to_datetime(tracks['release_date'], errors='coerce')

# Check if any NaT (errors) remain
print(f"Remaining invalid dates: {tracks['release_date'].isna().sum()}")

Remaining invalid dates: 0


In [26]:
# --- STEP 1: Transform duration_ms in duration_min ---
tracks['duration_ms'] = tracks['duration_ms'] / (1000 * 60)
# --- STEP 2: Rename duration_ms in duration_min ---
tracks = tracks.rename(columns={'duration_ms': 'duration_min'})
tracks.duration_min.head()

0    2.115050
1    1.636667
2    3.027333
3    2.948450
4    2.718000
Name: duration_min, dtype: float64

In [43]:
print(f"""Key unique values:
{sorted(tracks['key'].unique().tolist())}
Mode unique values:
{sorted(tracks['mode'].unique().tolist())}""")

Key unique values:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Mode unique values:
[0, 1]


`key` integer
The key the track is in. Integers map to pitches using standard [Pitch Class](https://en.wikipedia.org/wiki/Pitch_class) notation. E.g. 0 = C, 1 = C♯/D♭, 2 = D, and so on. If no key was detected, the value is -1.

In [46]:
url = 'https://en.wikipedia.org/wiki/Pitch_class'

headers = {
    'User-Agent': 'SpotifyAnalysisProject/1.0 (manuel.cernigoj@gmail.com)'
    }
page = requests.get(url, headers=headers)
print(f"""Page status code: {page.status_code}
Page preview: 
{page.text[:500]}""")

Page status code: 200
Page preview: 
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled wp25easte


In [65]:
soup = BeautifulSoup(page.content, "html.parser")
table = soup.find(class_="wikitable")
rows = table.find_all('tr')

In [78]:
pitch_classes = []

for row in rows[1:]:
    th = row.find('th')
    td = row.find('td')
    
    pitch = th.text.strip().split(',')[0].strip()
    tonal = td.text.strip().split(',')[0].strip()
    
    pitch_classes.append({
        'pitch': pitch,
        'key_name': tonal
    })
pitch_classes

[{'pitch': '0', 'key_name': 'C'},
 {'pitch': '1', 'key_name': 'C♯'},
 {'pitch': '2', 'key_name': 'D'},
 {'pitch': '3', 'key_name': 'D♯'},
 {'pitch': '4', 'key_name': 'E'},
 {'pitch': '5', 'key_name': 'F'},
 {'pitch': '6', 'key_name': 'F♯'},
 {'pitch': '7', 'key_name': 'G'},
 {'pitch': '8', 'key_name': 'G♯'},
 {'pitch': '9', 'key_name': 'A'},
 {'pitch': '10', 'key_name': 'A♯'},
 {'pitch': '11', 'key_name': 'B'}]

In [79]:
df_keys = pd.DataFrame(pitch_classes)
df_keys

,pitch,key_name
0,0,C
1,1,C♯
2,2,D
3,3,D♯
4,4,E
5,5,F
6,6,F♯
7,7,G
8,8,G♯
9,9,A


In [83]:
df_keys.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   pitch     12 non-null     str  
 1   key_name  12 non-null     str  
dtypes: str(2)
memory usage: 324.0 bytes


In [85]:
df_keys['pitch'] = df_keys['pitch'].astype(int)
df_keys.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   pitch     12 non-null     int64
 1   key_name  12 non-null     str  
dtypes: int64(1), str(1)
memory usage: 324.0 bytes


In [86]:
tracks = tracks.merge(df_keys, how = 'left', left_on = 'key', right_on = 'pitch')
tracks.head()

,id,name,popularity,duration_min,explicit,artists,id_artists,release_date,danceability,energy,...,acousticness,instrumentalness,liveness,valence,tempo,time_signature,main_artist,id_main_artist,pitch,key_name
0,35iwgR4jXetI318WEWsa1Q,Carve,6,2.115050,0,[Uli],[45tIt06XoI0Iio4LBEVpls],1922-02-22,0.645,0.4450,...,0.674,0.7440,0.151,0.127,104.851,3,Uli,45tIt06XoI0Iio4LBEVpls,0,C
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,1.636667,0,[Fernando Pessoa],[14jtPCOoNZwquk5wd9DxrY],1922-06-01,0.695,0.2630,...,0.797,0.0000,0.148,0.655,102.009,1,Fernando Pessoa,14jtPCOoNZwquk5wd9DxrY,0,C
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,3.027333,0,[Ignacio Corsini],[5LiOoJbxVSAMkBS2fUm3X2],1922-03-21,0.434,0.1770,...,0.994,0.0218,0.212,0.457,130.418,5,Ignacio Corsini,5LiOoJbxVSAMkBS2fUm3X2,1,C♯
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,2.948450,0,[Ignacio Corsini],[5LiOoJbxVSAMkBS2fUm3X2],1922-03-21,0.321,0.0946,...,0.995,0.9180,0.104,0.397,169.980,3,Ignacio Corsini,5LiOoJbxVSAMkBS2fUm3X2,7,G
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,2.718000,0,[Dick Haymes],[3BiJGZsyX9sJchTqcSA7Su],1922-01-01,0.402,0.1580,...,0.989,0.1300,0.311,0.196,103.220,4,Dick Haymes,3BiJGZsyX9sJchTqcSA7Su,3,D♯


In [32]:
# Extract year
tracks['year'] = tracks['release_date'].dt.year

In [27]:
# Drop any duplicate record to avoid distortions
tracks = tracks.drop_duplicates(subset=['id'])

In [28]:
print(tracks.info())
tracks.head()

<class 'pandas.DataFrame'>
RangeIndex: 586672 entries, 0 to 586671
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                586672 non-null  str           
 1   name              586672 non-null  str           
 2   popularity        586672 non-null  int64         
 3   duration_min      586672 non-null  float64       
 4   explicit          586672 non-null  int64         
 5   artists           586672 non-null  object        
 6   id_artists        586672 non-null  object        
 7   release_date      586672 non-null  datetime64[us]
 8   danceability      586672 non-null  float64       
 9   energy            586672 non-null  float64       
 10  key               586672 non-null  int64         
 11  loudness          586672 non-null  float64       
 12  mode              586672 non-null  int64         
 13  speechiness       586672 non-null  float64       
 14  acousticness   

,id,name,popularity,duration_min,explicit,artists,id_artists,release_date,danceability,energy,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,main_artist,id_main_artist
0,35iwgR4jXetI318WEWsa1Q,Carve,6,2.115050,0,[Uli],[45tIt06XoI0Iio4LBEVpls],1922-02-22,0.645,0.4450,...,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3,Uli,45tIt06XoI0Iio4LBEVpls
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,1.636667,0,[Fernando Pessoa],[14jtPCOoNZwquk5wd9DxrY],1922-06-01,0.695,0.2630,...,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1,Fernando Pessoa,14jtPCOoNZwquk5wd9DxrY
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,3.027333,0,[Ignacio Corsini],[5LiOoJbxVSAMkBS2fUm3X2],1922-03-21,0.434,0.1770,...,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5,Ignacio Corsini,5LiOoJbxVSAMkBS2fUm3X2
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,2.948450,0,[Ignacio Corsini],[5LiOoJbxVSAMkBS2fUm3X2],1922-03-21,0.321,0.0946,...,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3,Ignacio Corsini,5LiOoJbxVSAMkBS2fUm3X2
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,2.718000,0,[Dick Haymes],[3BiJGZsyX9sJchTqcSA7Su],1922-01-01,0.402,0.1580,...,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4,Dick Haymes,3BiJGZsyX9sJchTqcSA7Su


In [ ]:
features = [
    'tempo', 'loudness', 'energy', 'danceability',
    'valence', 'acousticness', 'liveness',
    'speechiness', 'instrumentalness'
]

### 4. Data Export

In [35]:
tracks.to_csv("../data/tracks_clean.csv", index=False)